# 03 — Runnables (LCEL, from the ground up)
### LangChain Foundations

Every notebook so far has quietly used `prompt | model`. That `|` isn't Python's bitwise-or doing
something clever by accident — it's a deliberately designed interface called the **Runnable
protocol**, and it's the single idea LangChain (LCEL — "LangChain Expression Language") is built on.

The goal of this notebook is narrow and specific: **build a minimal version of it yourself**, so
`|` stops being magic before you start leaning on the real thing everywhere.

## What's actually broken about "just write your own classes"

You've already tried the alternative. Look back at whatever version of `DummyPromptTemplate`,
`DummyLlm`, and a hand-rolled `Chain` class you may have written before reaching for LangChain
properly. It works — for exactly the one shape you built it for. The moment you want:

- **two steps running in parallel** instead of one after another,
- **a decision** ("if this is a billing question, use prompt A; otherwise, prompt B"),
- **streaming** the final output token-by-token,
- **batching** 50 inputs through the same pipeline efficiently,

...you're back to writing a brand new bespoke class for each case, because `.run()` only ever meant
"do the one sequential thing this class was written for." Nothing about it is *composable*.

That's the actual problem `Runnable` solves — not "prompt | model looks cool," but "every single
component speaks the same four-verb language (`invoke` / `batch` / `stream` / `ainvoke`), so they
can be wired together in *any* shape without writing a new glue class every time."

## Setup

In [1]:
# %pip install -q langchain langchain-core langchain-groq python-dotenv


In [2]:
import os, time
from dotenv import load_dotenv
load_dotenv()
print("Environment ready.")


Environment ready.


---
## 1. Build the minimal version yourself

This is the whole trick of `Runnable`: **one base class with one required method (`invoke`), plus
one operator overload (`__or__`) that chains two Runnables together into a third Runnable.** That's
genuinely most of it. Everything LangChain ships (`RunnableLambda`, `RunnableParallel`,
`RunnableBranch`, chat models, prompt templates...) is just a class that implements `invoke` and
gets this chaining behavior for free.

In [3]:
class MiniRunnable:
    """A deliberately tiny version of LangChain's Runnable, built to show what `|` actually does."""

    def invoke(self, input):
        raise NotImplementedError("Subclasses must implement invoke()")

    def __or__(self, other):
        # This is the entire trick behind `prompt | model | parser`.
        # Python calls __or__ whenever you write `a | b` — we hijack that to mean "chain these".
        return MiniRunnableSequence(self, other)


class MiniRunnableSequence(MiniRunnable):
    """The result of `a | b` — itself a Runnable, so you can keep chaining: `a | b | c`."""

    def __init__(self, first, second):
        self.first = first
        self.second = second

    def invoke(self, input):
        intermediate = self.first.invoke(input)
        return self.second.invoke(intermediate)


class MiniRunnableLambda(MiniRunnable):
    """Wraps any plain Python function so it can join the pipeline."""

    def __init__(self, func):
        self.func = func

    def invoke(self, input):
        return self.func(input)


In [4]:
# Three tiny steps, none of which know about each other, chained with plain `|`:
add_greeting = MiniRunnableLambda(lambda name: f"Hello, {name}!")
shout        = MiniRunnableLambda(lambda text: text.upper())
add_emoji    = MiniRunnableLambda(lambda text: f"{text} 👋")

pipeline = add_greeting | shout | add_emoji   # <-- exactly the syntax you've used all along

print(pipeline.invoke("Deepak"))
print(type(pipeline))   # a MiniRunnableSequence — the chain IS a Runnable too, chainable further


HELLO, DEEPAK! 👋
<class '__main__.MiniRunnableSequence'>


That's genuinely it. `pipeline` is a `MiniRunnableSequence` holding `add_greeting` and
`(shout | add_emoji)` nested inside it — each `|` just builds one more layer of "run this, then
feed the result into that." Real LangChain adds a lot on top (async, streaming, batching,
parallel branches, error handling, tracing) — but the core wiring you just wrote **is** the core
wiring LangChain uses. Nothing above was a simplification of a different idea; it's the same idea.

---
## 2. The real thing: `RunnableLambda`

Same concept as `MiniRunnableLambda` above, production-grade. Wrap any plain Python function and it
joins the pipeline — this is how your own custom logic (cleaning text, calling an external API,
reshaping data) slots in next to prompts and models.

In [5]:
from langchain_core.runnables import RunnableLambda

count_words = RunnableLambda(lambda text: len(text.split()))

print(count_words.invoke("this sentence has five words"))


5


---
## 3. `RunnableSequence` via `|` — a real prompt → model → parser chain

This is the pattern you've been using since `01_Models`, now with a proper output parser on the end
so you get a clean string back instead of an `AIMessage` object.

In [7]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in one sentence."),
    ("human", "{question}"),
])
model = ChatGroq(
    model="openai/gpt-oss-120b",  # Higher rate limits than the 70B model
    #max_retries=3
)
parser = StrOutputParser()

chain = prompt | model | parser   # three completely different classes, one shared interface

result = chain.invoke({"question": "Why is the sky blue?"})
print(result, "\n(type:", type(result).__name__, ")")   # a plain str, not an AIMessage


Because molecules in the atmosphere scatter shorter (blue) wavelengths of sunlight more efficiently than longer (red) wavelengths, giving the sky its blue color. 
(type: TextAccessor )


---
## 4. `RunnableParallel` — running independent steps at the same time

When two steps don't depend on each other's output, running them sequentially just wastes time.
`RunnableParallel` (a dict of Runnables) runs every branch concurrently and returns a dict of results.

**Real-world case:** a new support ticket needs both a one-line summary *and* an urgency
classification. Neither depends on the other — perfect candidate for parallel execution.

In [8]:
from langchain_core.runnables import RunnableParallel

summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the customer's issue in under 10 words."),
    ("human", "{ticket}"),
])
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify urgency as exactly one word: Low, Medium, or High. Reply with just that word."),
    ("human", "{ticket}"),
])

ticket_analysis = RunnableParallel(
    summary=summarize_prompt | model | parser,
    urgency=classify_prompt | model | parser,
)

ticket = "My card was charged three times for the same order and I need this fixed today."

start = time.time()
result = ticket_analysis.invoke({"ticket": ticket})
print(result)
print(f"\n--- {time.time() - start:.2f}s for BOTH calls (they ran concurrently, not one after the other) ---")


{'summary': 'Card charged three times for same order, urgent fix.', 'urgency': 'High'}

--- 1.13s for BOTH calls (they ran concurrently, not one after the other) ---


---
## 5. `RunnablePassthrough` — keeping the original input alongside new results

A common problem: step 2 of a chain needs BOTH the original input AND the output of step 1. Plain
`RunnableSequence` only hands step 2 whatever step 1 returned — the original input is gone by then.
`RunnablePassthrough.assign()` fixes this: it keeps every existing key and *adds* new ones.

In [9]:
from langchain_core.runnables import RunnablePassthrough

# .assign() takes the input dict, runs each named Runnable against it, and ADDS the result
# as a new key — the original keys (like "ticket") survive into the next step.
enrich = RunnablePassthrough.assign(
    urgency=classify_prompt | model | parser,
)

enriched = enrich.invoke({"ticket": ticket})
print(enriched.keys())          # dict_keys(['ticket', 'urgency']) — original input preserved!
print(enriched["ticket"][:40], "...")
print(enriched["urgency"])


dict_keys(['ticket', 'urgency'])
My card was charged three times for the  ...
High


This is exactly the pattern you'll meet again in `09_Retriever/`: a RAG chain needs the
original user question available at the final prompt step, *alongside* the retrieved documents —
`RunnablePassthrough` is how the question survives that far.

---
## 6. `RunnableBranch` — conditional routing

Different inputs sometimes need genuinely different handling, not just a different prompt string.
`RunnableBranch` takes a list of `(condition, runnable)` pairs, checks them in order, and runs the
first one that matches — with a required default at the end.

In [10]:
from langchain_core.runnables import RunnableBranch

billing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a billing specialist. Be precise about numbers and next steps."),
    ("human", "{ticket}"),
])
technical_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical support specialist. Ask for device/OS details if missing."),
    ("human", "{ticket}"),
])
general_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a general support agent."),
    ("human", "{ticket}"),
])

route_by_category = RunnableBranch(
    (lambda x: x["category"] == "Billing",   billing_prompt   | model | parser),
    (lambda x: x["category"] == "Technical", technical_prompt | model | parser),
    general_prompt | model | parser,   # default — required, runs if nothing above matched
)

print(route_by_category.invoke({"category": "Billing",   "ticket": "I was double-charged this month."}))
print()
print(route_by_category.invoke({"category": "Technical", "ticket": "App crashes on startup."}))


I’m sorry to hear that you were double‑charged. To investigate and resolve this quickly, could you please provide the following details?

| Information needed | Why it helps |
|--------------------|--------------|
| **Account / Customer ID** (or the email/phone you use for your account) | Locate your billing record |
| **Invoice or statement date** (e.g., “April 2026”) | Identify the specific billing cycle |
| **Amount shown for each charge** (e.g., $49.99 × 2) | Verify the duplicate amount |
| **Transaction IDs or last 4 digits of the card used** (if available) | Match the charges in our payment processor |
| **Date(s) the charges appeared on your card statement** | Confirm timing and avoid confusion with other fees |

**Next steps once I have that information**

1. **Verify the duplicate** in our billing system.  
2. **Issue a refund** for the extra charge (full amount, plus any applicable fees).  
3. **Send you a confirmation email** with the refund reference number and an estimated

---
## 7. Batch and stream — free capabilities, no extra code

Because every Runnable speaks the same four-verb interface, **`.batch()` and `.stream()` work on
the whole chain you built above** — not just on a raw model call like in `01_Models`.

In [11]:
# .batch() — process several inputs efficiently in one call instead of a Python for-loop
questions = [{"question": "What causes rain?"}, {"question": "Why do leaves change color?"}]
results = chain.batch(questions)
for q, r in zip(questions, results):
    print(f"{q['question']} -> {r}")


What causes rain? -> Rain is caused when warm, moist air rises, cools, and condenses into water droplets that coalesce and fall under gravity.
Why do leaves change color? -> Leaves change color because the breakdown of chlorophyll in autumn reveals underlying pigments like carotenoids and anthocyanins, which become visible as the green fades.


In [12]:
# .stream() — works on the FULL chain, including the parser at the end
for token in chain.stream({"question": "Explain gravity briefly."}):
    print(token, end="", flush=True)


Gravity is the attractive force that masses exert on each other, causing objects to accelerate toward one another, most noticeably pulling things toward the Earth’s center.

---
## Real-world capstone: a support-ticket triage pipeline

Every piece from this notebook, combined into one pipeline:

1. **`RunnableParallel`** — summarize and classify the ticket at the same time
2. **`RunnablePassthrough`-style key preservation** — keep the original ticket text alongside those results
3. **`RunnableBranch`** — route to a category-specific reply prompt based on the classification
4. One `.invoke()` call runs the entire thing.

In [13]:
# Step 1: classify + summarize in parallel, keeping the original ticket text alongside them
classify_category_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify this ticket as exactly one word: Billing, Technical, or General."),
    ("human", "{ticket}"),
])

analyze = RunnablePassthrough.assign(
    summary=summarize_prompt | model | parser,
    category=classify_category_prompt | model | parser,
)

# Step 2: route to a category-specific reply, using the category we just computed
reply_router = RunnableBranch(
    (lambda x: "Billing" in x["category"],   billing_prompt   | model | parser),
    (lambda x: "Technical" in x["category"], technical_prompt | model | parser),
    general_prompt | model | parser,
)

# Step 3: wire it together — analyze first, THEN route based on what analyze produced
triage_pipeline = analyze | RunnablePassthrough.assign(reply=reply_router)

result = triage_pipeline.invoke({"ticket": "I can't log in and I've tried resetting my password twice."})

print("Summary :", result["summary"])
print("Category:", result["category"])
print("Reply   :", result["reply"])


Summary : Unable to log in after multiple password resets.
Category: Technical
Reply   : I’m sorry you’re having trouble logging in. To help pinpoint the issue, could you let me know a few details?

1. **What service or app are you trying to access?** (e.g., our web portal, mobile app, desktop client)  
2. **Which device and operating system are you using?** (e.g., iPhone 15 iOS 17, Android 13, Windows 11, macOS 14, etc.)  
3. **Are you seeing any error messages or codes when you try to sign in?** If so, please copy the exact wording.  
4. **Did you receive a password‑reset confirmation email and were you able to set a new password successfully?**  
5. **Do you use any two‑factor authentication (SMS, authenticator app, etc.)?**  

With this information I can give you the most accurate steps to get you back into your account.


### Try it yourself
1. Feed the triage pipeline a clearly billing-related ticket and a clearly general one — confirm the
   `category` and the tone of `reply` both shift correctly.
2. Add a fourth branch (e.g. `"Feature Request"`) to `reply_router`, with its own prompt.
3. Time `triage_pipeline.invoke()` vs. writing the same 3 LLM calls sequentially by hand (no
   `RunnableParallel`) — confirm the parallel version is meaningfully faster.
4. Swap `model` for the OpenAI or Gemini model from `01_Models/01_chat_models.ipynb` — nothing else
   in this notebook needs to change. That portability is the entire point of the Runnable interface.

---
**Next:** `04_Chains/` — now that you understand what's happening under the hood, you'll use these
same building blocks (sequential, parallel, conditional) as first-class LangChain concepts, without
having to assemble them from primitives every time.
